### Build Chroma Multi-Modal Index with LlamaIndex

Tutorial: https://docs.llamaindex.ai/en/stable/examples/multi_modal/ChromaMultiModalDemo/

In [65]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext
from chromadb.utils.embedding_functions import OpenCLIPEmbeddingFunction
from llama_index.core.indices import MultiModalVectorStoreIndex
from chromadb.utils.data_loaders import ImageLoader
import chromadb
import numpy as np
from PIL import Image
from llama_index.multi_modal_llms.ollama import OllamaMultiModal
from llama_index.core import SimpleDirectoryReader

from llama_index.core.prompts import PromptTemplate
from llama_index.core.query_engine import SimpleMultiModalQueryEngine

In [11]:
from llama_index.embeddings.clip import ClipEmbedding

embeder =ClipEmbedding()


100%|███████████████████████████████████████| 338M/338M [03:20<00:00, 1.77MiB/s]


In [13]:
text_embedding= embeder.get_text_embedding("hello")
print(len(text_embedding))

512


In [ ]:
image_embedding= embeder.get_image_embedding("../data/2.jpg")
print(len(image_embedding))

512


### Set the embedding mode

In [24]:
# set defalut text and image embedding functions
embedding_function = OpenCLIPEmbeddingFunction()

# DataLoader for images    
image_loader = ImageLoader()

### Create empty collection

In [28]:
# Create a persistent client
client = chromadb.PersistentClient(path="../multimodal_vectorstore_2")

# Create a multimodal collection
collection = client.get_or_create_collection(
    name="multimodal_collection", 
    embedding_function=embedding_function,
    data_loader=image_loader,
    )

### Load images in a list of ImageDocuments Objects

In [29]:
# load documents
documents = SimpleDirectoryReader("../data").load_data()
documents

[ImageDocument(id_='4f8e687c-c481-4a73-b2d7-a2b9feb4d20b', embedding=None, metadata={'file_path': 'd:\\workspace\\proyecto_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\data\\11.jpg', 'file_name': '11.jpg', 'file_type': 'image/jpeg', 'file_size': 627983, 'creation_date': '2024-11-12', 'last_modified_date': '2024-10-26'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, text='', mimetype='text/plain', start_char_idx=None, end_char_idx=None, text_template='{metadata_str}\n\n{content}', metadata_template='{key}: {value}', metadata_seperator='\n', image=None, image_path='d:\\workspace\\proyecto_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\data\\11.jpg', image_url=None, image_mimetype=None, text_embedding=None),
 ImageDocument(id_='e309a5b6-64da-40fd-b6e8-efc

### Create Chroma Vector Store 

In [30]:
# set up ChromaVectorStore and load in data with LlamaIndex 
vector_store = ChromaVectorStore(chroma_collection=collection)
vector_store

ChromaVectorStore(stores_text=True, is_embedding_query=True, flat_metadata=True, collection_name=None, host=None, port=None, ssl=False, headers=None, persist_dir=None, collection_kwargs={})

In [31]:
# create storage context
storage_context = StorageContext.from_defaults(vector_store=vector_store)
storage_context

StorageContext(docstore=<llama_index.core.storage.docstore.simple_docstore.SimpleDocumentStore object at 0x0000012AE2162A50>, index_store=<llama_index.core.storage.index_store.simple_index_store.SimpleIndexStore object at 0x0000012AE40BBA10>, vector_stores={'default': ChromaVectorStore(stores_text=True, is_embedding_query=True, flat_metadata=True, collection_name=None, host=None, port=None, ssl=False, headers=None, persist_dir=None, collection_kwargs={}), 'image': SimpleVectorStore(stores_text=False, is_embedding_query=True, data=SimpleVectorStoreData(embedding_dict={}, text_id_to_ref_doc_id={}, metadata_dict={}))}, graph_store=<llama_index.core.graph_stores.simple.SimpleGraphStore object at 0x0000012AE40B9A00>, property_graph_store=None)

### Create VectorStoreIndex

In [53]:
# create index of documents
index = VectorStoreIndex.from_documents(
    documents = documents,
    storage_context=storage_context,
    embed_model=embeder
)

In [54]:
index

In [55]:
retriever = index.as_retriever(similarity_top_k=1)

`Caso 1`: consulta de texto.

In [56]:
retrival_result = retriever.retrieve("Sandal")
retrival_result

[NodeWithScore(node=ImageNode(id_='660f6037-d5f8-4645-b3de-7e74dd1c7a42', embedding=None, metadata={'file_path': 'd:\\workspace\\proyecto_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\data\\5_small.jpg', 'file_name': '5_small.jpg', 'file_type': 'image/jpeg', 'file_size': 6623, 'creation_date': '2024-11-12', 'last_modified_date': '2024-11-12'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='9a60b34b-7386-4c04-9192-69d6b9f31c67', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'd:\\workspace\\proyecto_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\data\\5_small.jpg', 'file_name': '5_small.jpg', 'file_type': 'image/jpeg', 'file_size': 6623, 'creation_date': '2024-11-12', 'last_modified_date'

### Create MultiModalVectorStoreIndex

In [ ]:
# Create a persistent client
client_2 = chromadb.PersistentClient(path="../multimodal_vectorstore_5")

# Create a multimodal collection
collection_2 = client_2.get_or_create_collection(
    name="multimodal_collection", 
    embedding_function=embedding_function,
    data_loader=image_loader,
    )

# set up ChromaVectorStore and load in data with LlamaIndex 
vector_store_2 = ChromaVectorStore(chroma_collection=collection_2)

# create storage context
storage_context_2 = StorageContext.from_defaults(vector_store=vector_store_2)

# create index of documents
index_2 = MultiModalVectorStoreIndex.from_documents(
    documents = documents,
    image_embeded_model=embeder,
    storage_context=storage_context_2
)

index_2

### QueryEngine

In [62]:
mm_model = OllamaMultiModal(model="llava:7b", request_timeout=60.0)
query_engine= index_2.as_query_engine(llm=mm_model)
query_engine


In [67]:
query_engine.retrieve("Que tipo de botas hay ")

[]

In [66]:
query_engine.query("Que tipo de botas hay en la imagen")

Response(response=' Lo siento, no puedo ayudarte con esta solicitud, ya que la imagen que se ha proporcionado no está disponible para mí. Si puedes proporcionarme la imagen o describir su contenido en texto, estaría encantado de poder ayudarte. ', source_nodes=[], metadata={'text_nodes': [], 'image_nodes': []})

In [64]:
query_engine.image_query(image_path="../data/2.jpg", prompt_str="Describe en pocas palabras esta imagen en español de forma precisa y concisa ")

Response(response=' En este momento no puedo proporcionar información sobre la imagen que se menciona ya que carece de un contexto para poder realizar una descripción. ', source_nodes=[], metadata={'image_nodes': []})

In [59]:
retriever_2 = index_2.as_retriever()
retriever_2


In [ ]:
retrival_result_2 = retriever_2.retrieve("boots")
retrival_result_2